In [12]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder
from structural_analysis.Material import Material

# Loading the pre-computed planforms and the fuselage

In [13]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [14]:
# --- Import Onshape pull utilities ---
sys.path.append(os.path.abspath(os.getcwd()))
from onshape_pull import (
    fetch_variable_studio, fetch_measurement_features,
    evaluate_measurements, load_cached_masses, fetch_mass_properties,
    compute_cg_scenarios, lookup_var, lookup_meas,
    UPDATE_MASSES,
)

# --- Pull data from Onshape ---
variables = fetch_variable_studio()
meas_names = fetch_measurement_features()
measurements = evaluate_measurements(meas_names)
components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
cg_data = compute_cg_scenarios(components)

# --- Z offset (axle datum) ---
Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
            + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# --- Build drag components ---
engine_bay = Bay(
    surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
    length=0.172,                       # 172 mm (hardcoded, not in Onshape)
    diameter=lookup_var(variables, "engine_diameter")[0],
)

Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
nose_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
)

Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
main_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
)

fuselage = Fuselage(
    surface_wetted=lookup_meas(measurements, "Wetted_Area"),
    length_total=lookup_var(variables, "FuselageLength")[0],
    diameter_max=lookup_var(variables, "FuselageHeight")[0],
    upsweep=0.0,
    base_area=lookup_meas(measurements, "Base_Area"),
)

# --- X-position helpers ---
WingPortDistance, _, _ = lookup_var(variables, "WingPortDistance")
WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

# --- Fixed parameters ---
fixed = Fixed(
    mass=cg_data["mass"],
    fuel_mass=cg_data["fuel_mass"],
    x_cg_min=cg_data["x_cg_min"],
    x_cg_max=cg_data["x_cg_max"],
    x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
    z_cg=cg_data["z_cg_full"] + z_offset,
    z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
    z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
    x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
    x_LE_wing=WingPortDistance + 0.115 - WingPortWidth / 2,
    x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
    x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
    x_main_gear=lookup_meas(measurements, "x_main_gear"),
    y_main_gear=0.419,
    fuselage=fuselage,
    nose_gear=nose_gear,
    main_gear=main_gear,
    engine_bay=engine_bay,
)

  → Found 78 occurrences, fetching mass properties...
    ... processed 10/78 occurrences
    ... processed 20/78 occurrences
    ... processed 30/78 occurrences
    ... processed 40/78 occurrences
    ... processed 50/78 occurrences
    ... processed 60/78 occurrences
    ... processed 70/78 occurrences
  → Mass data cached to mass_cache.json


In [15]:
with open("pickles/fixed_pickle.pcl", "wb") as f:
    pickle.dump(fixed, f)

In [16]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [17]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [18]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

In [19]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(plaforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=max(4., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_c=max(5., main_wing.aspect_ratio/2))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

0.2944239494738869
Stresses 509648.13360608864, 1564471.2029410438, 0.0004
0.294423949473887
Stresses 1543256.6835347144, 16087455.390532745, 0.0004
0.49852996601511185
Stresses 756431.0281907781, 2265769.357635026, 0.0004
0.49852996601511185
Stresses 1757504.6274197653, 13029234.885097735, 0.0004
0.47147433321752963
Stresses 725434.806093215, 2174098.8304528873, 0.0004
0.47147433321752963
Stresses 1733491.8281176954, 13282545.737303775, 0.0004
0.4743830862492583
Stresses 728788.1547705199, 2188490.511811628, 0.0004
0.47438308624925807
Stresses 1736122.8693930344, 13281355.349121315, 0.0004
0.4740628788530693
Stresses 728419.2568968781, 2187963.554643, 0.0004
0.4740628788530693
Stresses 1735833.8343932386, 13287954.828071218, 0.0004
0.474080457959521
Stresses 728439.5106844445, 2187064.1297863848, 0.0004
0.474080457959521
Stresses 1735849.7060057647, 13281903.540581586, 0.0004
0.3415475767479746
Stresses 661665.8918851808, 2513038.8733308823, 0.0004
0.3412436975038259
Stresses 332337.3

In [20]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
ac_bad:Aircraft = aircraft[np.argmax(s_ratios)],
print(np.argmax(s_ratios))
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord)

[0.12691351826301694, 0.05102398098356617, 0.1269148056588388, 0.16334790549096415, 0.41565529331343243, 0.423313399354861, 0.23433307890951083, 0.2175752948674255, 0.12742433663899355, 0.050222830395130244, 0.12750696345879597, 0.16415521541475633, 0.4148773328589922, 0.42236861016588945, 0.2335027970291055, 0.2167004669491315, 0.12795681141488613, 0.0493872313808923, 0.12814212343360343, 0.1649972843890114, 0.4140656791700249, 0.42138347957266564, 0.23263648385036081, 0.2157883481827833, 0.21460784197517235, 0.13681324888379312, 0.21460784197517235, 0.3036464260712438, 0.3920451421038749, 0.4026265542454518, 0.27035053812637205, 0.26102742129212, 0.21072645640339516, 0.12947206082258247, 0.21072645640339516, 0.29626484986498275, 0.39882756209375275, 0.4109140468280077, 0.27743992343411306, 0.2689101599261266, 0.20904691116935847, 0.12630105457642615, 0.20904691116935847, 0.2930761277515688, 0.4017527799874088, 0.41450137706342227, 0.2804965759646967, 0.2723231623499744, 0.27039318401

# Checking if reuirements are met

In [10]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [11]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

Fuel available: 11.000000001437998 kg
Fuel required: 7.009019028012672 kg
Difference: 3.9909809734253265 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.022578454835737 kg
Difference: 3.977421546602261 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.172962019260426 kg
Difference: 3.827037982177572 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.110699205206417 kg
Difference: 3.8893007962315815 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 6.546244112929252 kg
Difference: 4.453755888508746 kg
all constraints satisfied
ac mass: 36.17975845823961, 0.7845238526961408
MainWing: AR=5.0, tc=0.06, sweep=14.999999999999998 deg, cmac=-0.15
Failed: ['Empennage Requirement']

Fuel available: 11.000000001437998 kg
Fuel required: 6.897015420418916 kg
Difference: 4.102984581019082 kg
all constraints satisfied
ac mass: 36.17975845823961, 0.7845238526961408
Mai